# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Experimento toy Qwen condicionado por definiciones Markdown

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

El ejercicio usa el modelo conversacional oficial `Qwen/Qwen3-0.6B` [1] y LoRA [2]. El archivo Markdown completo con las definiciones se incorpora como mensaje de sistema en todos los ejemplos. La pérdida se calcula solo sobre el objeto de respuesta y la decodificación se restringe por un trie de tokens a los cinco JSON permitidos. Es un estudio toy aislado: no genera candidatos ni se incorpora a la selección de `03_07`.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA A100 de 40 GB** para Qwen. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. La celda detecta si falta el release exacto: en ese único caso lo obtiene desde GitHub —o mediante `local_upload`—, verifica todos sus SHA-256 y lo publica de forma atómica. Después promueve la copia activa cuando sea necesario; ya no requiere ejecutar `02_00` previamente. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [3]. La integridad del bundle se comprueba con SHA-256 [4]. No sincronice cachés de modelos ni entrene directamente sobre Drive; los cuadernos de entrenamiento Qwen copian cada checkpoint terminado mediante un TAR atómico y reanudable.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
import uuid
import zipfile

COLAB_NOTEBOOK_ID = "03_06b"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = False
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_AUTO_PUBLISH_MISSING_BUNDLE = True
COLAB_BUNDLE_SOURCE = "github"  # "github" o "local_upload"
COLAB_GITHUB_REPOSITORY = "lkoc/Trabajo_PLN-MIA-Grupo4"
COLAB_GITHUB_REF = "main"
COLAB_GITHUB_BUNDLE_PATH = "resultados/colab_bundle"
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "45f01474e1dab847f6546a68609593c10a998a29187ef224e2599780dc039b78"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "51d06e6ad1642c34315cdd58d26cccb3241ae9d1401133e1bad8b3bb840e2814"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _verify_expected_bundle(bundle_dir, expected_bundle_id=COLAB_NOTEBOOK_BUILD_BUNDLE_ID):
    manifest_path = Path(bundle_dir) / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta {manifest_path}")
    manifest = _read_manifest(manifest_path)
    computed_bundle_id = _bundle_id_for_manifest(manifest)
    if manifest.get("bundle_id") != computed_bundle_id:
        raise ValueError("bundle_manifest.json no contiene una identidad válida")
    if computed_bundle_id != expected_bundle_id:
        raise ValueError(
            f"Bundle inesperado: esperado={expected_bundle_id}, obtenido={computed_bundle_id}"
        )
    if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
        raise ValueError("El core del bundle no coincide con el fijado por este cuaderno")
    for name, expected_sha256 in _bundle_specs(manifest):
        artifact = Path(bundle_dir) / name
        if not artifact.is_file() or _sha256(artifact) != expected_sha256:
            raise ValueError(f"Artefacto ausente o inválido: {artifact}")
    return manifest

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if Path(manifest_path) != Path(bundle_dir) / "bundle_manifest.json":
        return False
    try:
        _verify_expected_bundle(bundle_dir, expected_bundle_id)
        return True
    except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _download_bundle_file(url, destination):
    destination = Path(destination)
    partial = destination.with_name(f".{destination.name}.partial")
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "ModeracionPeru-Colab-Bundle/2.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as target:
            while block := response.read(1024 * 1024):
                target.write(block)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()

def _prepare_bundle_staging():
    staging = Path("/content/moderacion_peru_bundle_source")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    return staging

def _uploaded_bundle_member(uploaded, expected_name):
    # Resuelve el nombre exacto o el sufijo (N) que agrega Colab al repetir una carga.
    if expected_name in uploaded:
        return expected_name
    suffix = Path(expected_name).suffix
    base = expected_name[:-len(suffix)] if suffix else expected_name
    prefix = f"{base} ("
    ending = f"){suffix}"
    candidates = []
    for actual_name in uploaded:
        if not actual_name.startswith(prefix) or not actual_name.endswith(ending):
            continue
        duplicate_number = actual_name[len(prefix):-len(ending)]
        if duplicate_number.isdigit():
            candidates.append(actual_name)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(
            f"La selección contiene varias copias de {expected_name}: {sorted(candidates)}"
        )
    return None

def _acquire_expected_bundle():
    staging = _prepare_bundle_staging()
    if COLAB_BUNDLE_SOURCE == "github":
        encoded_ref = urllib.parse.quote(COLAB_GITHUB_REF, safe="")
        base = (
            f"https://raw.githubusercontent.com/{COLAB_GITHUB_REPOSITORY}/"
            f"{encoded_ref}/{COLAB_GITHUB_BUNDLE_PATH}"
        )
        cache_key = urllib.parse.quote(COLAB_NOTEBOOK_BUILD_BUNDLE_ID, safe="")
        manifest_path = staging / "bundle_manifest.json"
        _download_bundle_file(
            f"{base}/bundle_manifest.json?bundle_id={cache_key}", manifest_path
        )
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            raise ValueError("El manifiesto descargado desde GitHub no es válido")
        if manifest["bundle_id"] != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError(
                "GitHub todavía no contiene el bundle fijado por este cuaderno. "
                "Sincronice resultados/colab_bundle o use COLAB_BUNDLE_SOURCE='local_upload'."
            )
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("GitHub contiene un project_core.zip distinto al esperado")
        for name, _ in _bundle_specs(manifest):
            encoded_name = urllib.parse.quote(name, safe="")
            _download_bundle_file(
                f"{base}/{encoded_name}?bundle_id={cache_key}", staging / name
            )
    elif COLAB_BUNDLE_SOURCE == "local_upload":
        from google.colab import files

        uploaded = files.upload()
        manifest_upload = _uploaded_bundle_member(uploaded, "bundle_manifest.json")
        if manifest_upload is None:
            raise FileNotFoundError("La selección no incluyó bundle_manifest.json")
        (staging / "bundle_manifest.json").write_bytes(uploaded[manifest_upload])
        manifest = _read_manifest(staging / "bundle_manifest.json")
        if manifest.get("bundle_id") != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError("Los archivos seleccionados no pertenecen al bundle esperado")
        required = {"bundle_manifest.json", *(name for name, _ in _bundle_specs(manifest))}
        resolved = {
            name: _uploaded_bundle_member(uploaded, name)
            for name in required - {"bundle_manifest.json"}
        }
        missing = sorted(name for name, actual_name in resolved.items() if actual_name is None)
        if missing:
            raise FileNotFoundError(f"Faltaron archivos del bundle: {missing}")
        for name, actual_name in resolved.items():
            (staging / name).write_bytes(uploaded[actual_name])
    else:
        raise ValueError("COLAB_BUNDLE_SOURCE debe ser 'github' o 'local_upload'")
    return staging, _verify_expected_bundle(staging)

def _write_latest_pointer(releases_dir, release_dir, manifest):
    pointer = {
        "schema_version": "1.0.0",
        "bundle_id": manifest["bundle_id"],
        "core_sha256": manifest["core"]["sha256"],
        "manifest_sha256": _sha256(Path(release_dir) / "bundle_manifest.json"),
        "published_at": datetime.now(timezone.utc).isoformat(),
    }
    latest_path = Path(releases_dir) / "latest.json"
    partial = Path(releases_dir) / f".latest-{uuid.uuid4().hex}.json"
    partial.write_text(json.dumps(pointer, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(partial, latest_path)
    return pointer

def _publish_expected_bundle(staging, releases_dir):
    manifest = _verify_expected_bundle(staging)
    releases_dir = Path(releases_dir)
    releases_dir.mkdir(parents=True, exist_ok=True)
    release_dir = releases_dir / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if release_dir.exists():
        _verify_expected_bundle(release_dir)
        release_status = "already_present_and_verified"
    else:
        partial = releases_dir / f".{COLAB_NOTEBOOK_BUILD_BUNDLE_ID}.partial-{uuid.uuid4().hex}"
        partial.mkdir()
        try:
            for name, _ in _bundle_specs(manifest):
                shutil.copyfile(Path(staging) / name, partial / name)
            shutil.copyfile(
                Path(staging) / "bundle_manifest.json",
                partial / "bundle_manifest.json",
            )
            _verify_expected_bundle(partial)
            os.replace(partial, release_dir)
        finally:
            if partial.exists():
                shutil.rmtree(partial)
        release_status = "auto_published_and_verified"
    pointer = _write_latest_pointer(releases_dir, release_dir, manifest)
    return {
        "status": release_status,
        "release_dir": release_dir,
        "latest_pointer": pointer,
    }

def _ensure_expected_drive_release(releases_dir):
    release_dir = Path(releases_dir) / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if _bundle_is_current(
        release_dir,
        release_dir / "bundle_manifest.json",
        COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
    ):
        return {"status": "already_present_and_verified", "release_dir": release_dir}
    if not COLAB_AUTO_PUBLISH_MISSING_BUNDLE:
        raise RuntimeError(
            "Drive no contiene el release esperado y COLAB_AUTO_PUBLISH_MISSING_BUNDLE=False"
        )
    staging, _ = _acquire_expected_bundle()
    return _publish_expected_bundle(staging, releases_dir)

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    RELEASES_DIR.mkdir(parents=True, exist_ok=True)
    release_check = _ensure_expected_drive_release(RELEASES_DIR)
    latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    RELEASE_DIR = RELEASES_DIR / latest_bundle_id
    manifest = _verify_expected_bundle(RELEASE_DIR)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_matches_notebook = (
        latest_pointer.get("bundle_id") == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
        and latest_pointer.get("manifest_sha256") == _sha256(release_manifest_path)
    )
    if latest_matches_notebook:
        release_source = (
            "auto_published_from_" + COLAB_BUNDLE_SOURCE
            if release_check["status"] == "auto_published_and_verified"
            else "latest_pointer"
        )
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive después de "
                f"verificar o autopublicar {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'estado_del_release': release_check['status'],
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Prompt operacional vigente', {'ruta': OPERATIONAL_PROMPT, 'versión': '3.2.0'}, tone='success')


Mounted at /content/drive


estado,ya_estaba_actualizado
bundle_id,45f01474e1dab847f6546a68609593c10a998a29187ef224e2599780dc039b78
bundle_del_notebook_al_generarse,45f01474e1dab847f6546a68609593c10a998a29187ef224e2599780dc039b78
origen_del_release,latest_pointer
estado_del_release,already_present_and_verified
core_sha256,51d06e6ad1642c34315cdd58d26cccb3241ae9d1401133e1bad8b3bb840e2814
generado,2026-08-15T12:50:02.136097+00:00
versión_inmutable_drive,Ver detalle/content/drive/MyDrive/ModeracionPeru_Colab/bundle_releases/45f01474e1dab847f6546a68609593c10a998a29187ef224e2599780dc039b78


is_colab,Sí
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""auto"", ""device_name"": ""NVIDIA A100-SXM4-40GB"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 42405855232, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
nvidia_smi,"NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07"
cwd,/content
free_runtime_bytes,69776928768


notebook_id,03_06b
run_id,03_06b_working_v2_1
drive_root,/content/drive/MyDrive/ModeracionPeru_Colab
runtime_root,/content/moderacion_peru
project_root,/content/moderacion_peru/project
input_paths,"Ver detalle{ ""dataset_5_salidas"": ""/content/moderacion_peru/inputs/datos/model_ready/v2/dataset_5_salidas.jsonl"" }"
scratch_output_dir,/content/moderacion_peru/runs/03_06b/03_06b_working_v2_1
drive_run_dir,/content/drive/MyDrive/ModeracionPeru_Colab/runs/03_06b/03_06b_working_v2_1
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""cuda"", ""device_name"": ""NVIDIA A100-SXM4-40GB"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 42405855232, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
resumed,No


ruta,/content/moderacion_peru/project/config/prompt_operacional_ollama_v3_2.md
versión,3.2.0


## Procedimiento reproducible por corridas

Este cuaderno es un ejercicio toy independiente. No produce `candidate.json`, no restaura modelos ajenos y no participa en `03_07`.

1. **Partición declarada.** `80:20:20` se interpreta como pesos normalizados `4:1:1`, porque no son porcentajes válidos. El resultado exacto es 800 train, 200 validation y 200 test. Cada daño aporta 40/10/10 y `SEGURO` 640/160/160.
2. **Muestreo.** `RUN_BUILD_TOY_DATASET=True` recorre el snapshot, conserva solo ejemplos puros, selecciona aleatoriamente dentro de cada categoría/split con semilla fija y usa como máximo un chunk por video. Revise que el panel confirme 1.200 videos únicos y cero fuga.
3. **Entrenamiento.** En Colab seleccione A100 y mantenga `RUN_TRAIN_QWEN=True`. Qwen recibe como contexto completo `config/definiciones_dano_toy_03_06b.md`; LoRA se ajusta con 800 filas.
4. **Salida estructurada.** La generación no depende solo de una instrucción: un trie de tokens limita cada respuesta a cinco objetos JSON posibles con `chunk_id` literal y una categoría válida.
5. **Eficacia.** Validation y el test toy retenido se evalúan por separado. La métrica principal es `strict_macro_f1`; cualquier JSON inválido cuenta como error. También se guardan accuracy, balanced accuracy, tabla por categoría, predicciones y matriz de confusión.
6. **Reanudación.** Conserve dataset, Markdown, semilla e hiperparámetros. Una firma idéntica devuelve `status=noop`; use `FORCE_REBUILD` o `FORCE_RETRAIN` solo para una repetición deliberada.

Las métricas describen únicamente este conjunto enriquecido de 1.200 ejemplos; no estiman prevalencia natural ni superioridad frente a los modelos de `03_01`–`03_06`.

## Restauración reproducible del dataset

In [2]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


status,verified_in_colab
input_key,dataset_5_salidas
path,/content/moderacion_peru/inputs/datos/model_ready/v2/dataset_5_salidas.jsonl
bytes,225478048


## Configuración y ejecución

In [ ]:
from moderacion_peru.toy_prompt_sft import build_toy_prompt_dataset,train_and_evaluate_toy_qwen
SOURCE_DATA=COLAB_CONTEXT.input('dataset_5_salidas') if COLAB_CONTEXT else ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
DEFINITIONS=ROOT/'config/definiciones_dano_toy_03_06b.md'
OUTPUT_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'resultados/03_06b_toy'
TOY_DATA=OUTPUT_ROOT/'toy_dataset.jsonl'
DEVICE='cuda' if COLAB_CONTEXT else 'auto'
SEED=20260815
EPOCHS=3
MAX_LENGTH=1536
RUN_BUILD_TOY_DATASET=True
RUN_TRAIN_QWEN=True
FORCE_REBUILD=False
FORCE_RETRAIN=False

show_summary('Diseño independiente 03_06b',{
    'muestras':'1.200 = 960 SEGURO + 60 por cada uno de cuatro daños',
    'partición':'80:20:20 como pesos 4:1:1 = 800/200/200',
    'por_daño':'40 train, 10 validation, 10 test',
    'SEGURO':'640 train, 160 validation, 160 test',
    'contexto':DEFINITIONS,
    'modelo':'Qwen/Qwen3-0.6B + LoRA causal',
    'salida':'JSON restringido a cinco objetos válidos por trie de tokens',
    'métrica_principal':'strict_macro_f1 en test toy; JSON inválido cuenta como error',
    'comparación_03_07':'prohibida; no se genera candidate.json',
},tone='warning')
if RUN_BUILD_TOY_DATASET:
    toy_result=run_with_progress('Muestreo estratificado toy',build_toy_prompt_dataset,SOURCE_DATA,TOY_DATA,seed=SEED,force=FORCE_REBUILD,progress_unit='fila')
    show_result('Toy dataset verificado',toy_result,tone='success')
if RUN_TRAIN_QWEN:
    if not TOY_DATA.is_file():
        raise FileNotFoundError('Falta toy_dataset.jsonl; active primero RUN_BUILD_TOY_DATASET.')
    qwen_toy_result=run_with_progress('Qwen toy: LoRA + JSON restringido',train_and_evaluate_toy_qwen,TOY_DATA,DEFINITIONS,OUTPUT_ROOT,device=DEVICE,seed=SEED,epochs=EPOCHS,max_length=MAX_LENGTH,force=FORCE_RETRAIN,progress_unit='fila')
    show_result('Eficacia independiente de Qwen toy',qwen_toy_result,tone='success')
if not (RUN_BUILD_TOY_DATASET or RUN_TRAIN_QWEN):
    show_callout('Ejecución desactivada','Active el muestreo, el entrenamiento o ambos. Mantenga ambos True para una corrida completa en A100.',tone='neutral')

muestras,1.200 = 960 SEGURO + 60 por cada uno de cuatro daños
partición,80:20:20 como pesos 4:1:1 = 800/200/200
por_daño,"40 train, 10 validation, 10 test"
SEGURO,"640 train, 160 validation, 160 test"
contexto,/content/moderacion_peru/project/config/definiciones_dano_toy_03_06b.md
modelo,Qwen/Qwen3-0.6B + LoRA causal
salida,JSON restringido a cinco objetos válidos por trie de tokens
métrica_principal,strict_macro_f1 en test toy; JSON inválido cuenta como error
comparación_03_07,prohibida; no se genera candidate.json


Muestreo estratificado toy: 0fila [00:00, ?fila/s]

status,created
dataset_path,/content/moderacion_peru/runs/03_06b/03_06b_working_v2_1/toy_dataset.jsonl
manifest_path,/content/moderacion_peru/runs/03_06b/03_06b_working_v2_1/toy_dataset_manifest.json
rows,1200
unique_videos,1200
distribution,"Ver detalle{ ""train"": { ""SEGURO"": 640, ""RACISMO_DISCRIMINACION"": 40, ""ATAQUE_POR_GENERO_IDENTIDAD"": 40, ""ACOSO_AMENAZA"": 40, ""CONTENIDO_SEXUAL"": 40 }, ""validation"": { ""SEGURO"": 160, ""RACISMO_DISCRIMINACION"": 10, ""ATAQUE_POR_GENERO_IDENTIDAD"": 10, ""ACOSO_AMENAZA"": 10, ""CONTENIDO_SEXUAL"": 10 }, ""test"": { ""SEGURO"": 160, ""RACISMO_DISCRIMINACION"": 10, ""ATAQUE_POR_GENERO_IDENTIDAD"": 10, ""ACOSO_AMENAZA"": 10, ""CONTENIDO_SEXUAL"": 10 } }"
split_rows,"Ver detalle{ ""train"": 800, ""validation"": 200, ""test"": 200 }"
pure_single_label_rows,Sí
video_disjoint,Sí


Qwen toy: LoRA + JSON restringido: 0fila [00:00, ?fila/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 13.91 GiB. GPU 0 has a total capacity of 39.49 GiB of which 2.94 GiB is free. Including non-PyTorch memory, this process has 36.54 GiB memory in use. Of the allocated memory 22.07 GiB is allocated by PyTorch, and 13.96 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Publicación o checkpoint en Drive

El dataset toy, el adaptador, las predicciones y las métricas pueden copiarse como un resultado independiente. No se publica ningún candidato para `03_07`. Esta celda permite guardar manualmente el resultado verificable en Drive.

In [ ]:
PUBLISH_TO_DRIVE = True
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None and globals().get('AUTO_PUBLISH_CHECKPOINTS'):
    show_callout('Checkpoint automático activo', 'La recuperación, los checkpoints periódicos, Ctrl+C y cada cierre de campaña ya publican una copia verificable en Drive.', tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación manual desactivada', 'Los entrenamientos 03_x ya publican automáticamente al completar; active esta celda solo para repetir la publicación final.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] Qwen Team, "Model Card: Qwen/Qwen3-0.6B," Hugging Face Hub, revision 6130ef31402718485ca4d80a6234f70d9a4cf362, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-0.6B/tree/6130ef31402718485ca4d80a6234f70d9a4cf362

[2] E. J. Hu, Y. Shen, P. Wallis, et al., "LoRA: Low-Rank Adaptation of Large Language Models," in Proc. ICLR, 2022. [Online]. Available: https://openreview.net/forum?id=nZeVKeeFYf9

[3] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[4] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.